# TURN-Mapper — tune learned segmenter v2 on the dev set, then test once (CPU)

docs/ML_IMPROVEMENT_PLAN.md phase C. **Input:** output of `turn-segmenter-s2-finetune` (`turn_seg_s2.pt`). Accelerator: None. Internet: On.

Dev set: 16 procedural institutional floors, seed 2026, never used for training, drawn in 8 styles. The best config is chosen by the rule written in `scripts/tune_learned.py`, then run once on the 12 synthetic test cases.

In [ ]:
!git clone -q https://github.com/ZakiNabeel/Turn-Mapper.git /kaggle/working/Turn-Mapper
%cd /kaggle/working/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1 || true
!pip -q install -e ".[ml]"
!git log --oneline -1
import glob, os
os.environ['TURN_SEG_WEIGHTS'] = glob.glob('/kaggle/input/**/turn_seg_s2.pt', recursive=True)[0]
print(os.environ['TURN_SEG_WEIGHTS'])

In [ ]:
!python scripts/tune_learned.py --n 16 --out /kaggle/working/tune 2>&1 | grep -v Warning | tee /kaggle/working/tune_log.txt | grep -E "^(classical|v1|doors|BEST|DONE)" 

In [ ]:
import json
R = json.load(open('/kaggle/working/tune/tune_results.json'))
print('| config | dev score | rooms R | rooms P | doors R | doors P | routes | wall x |')
for n, r in R['dev'].items():
    d = r['dev']; print(f"| {n} | {r['dev_score']} | {d.get('room_recall')} | {d.get('room_precision')} | {d.get('door_recall')} | {d.get('door_precision')} | {d.get('routes')} | {d.get('wall_crossings')} |")
print('best:', R['best'])
print()
print('| test case | ' + ' | '.join(R['test']) + ' |  (rooms R/P, doors R/P, routes, wall x)')
names = list(R['test'])
for i, row in enumerate(R['test'][names[0]]):
    cells = []
    for n in names:
        m = R['test'][n][i]
        cells.append(f"{m['room_recall']}/{m['room_precision']}, {m['door_recall']}/{m['door_precision']}, {m['route_found']}, {m['wall_crossing_edges']}")
    print(f"| {row['layout']} {row['style']} | " + ' | '.join(cells) + ' |')